# 04 — Explore manual-pick clustering and compare the 153- and 157-event catalogues

This notebook follows directly from the MATLAB consolidation script:

```text
/Users/thompsong/Developer/KSCRocketSeismology/matlab/fireball/
consolidate_spacex_matlab_legacy_data_epoch_complete.m
```

That script reads the legacy MATLAB/GISMO products, including
`spacexplosion.mat` and, when available, `catalog.mat`, and writes durable
exports to:

```text
/Users/thompsong/Developer/KSCRocketSeismology/
08_fireball_paper/metadata/legacy_export/
```

The key exported products are:

```text
manual_arrival_picks.csv
legacy_infrasound_event_catalogue.csv
legacy_catalog_157_events.csv
```

The manual-pick table preserves the original Antelope/GISMO pick times and
channel identities. The two catalogue files represent different legacy
event constructions:

- a 153-event catalogue stored in `infrasoundEvent`;
- a 157-event catalogue stored in `catalogobj.ontime/offtime`.

This notebook uses Unix epoch seconds as the authoritative interchange time
representation and compares the two legacy catalogues directly with the
preserved manual picks.

In [1]:
# Standard project configuration and isolated output namespace
from pathlib import Path
import sys

NOTEBOOK_NAME = "04_reconcile_manual_event_catalogs.ipynb"
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT_HINT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT_HINT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import notebook_context

CTX = notebook_context(NOTEBOOK_NAME, start=CURRENT_DIR)
CONFIG = CTX.config
PROJECT_ROOT = CTX.project_root

# Every notebook writes only inside its own numerically coded namespace.
OUTPUT_DIR = CTX.output_dir
OUTPUT_DATA_DIR = CTX.data_dir
OUTPUT_FIGURE_DIR = CTX.figure_dir
OUTPUT_LOG_DIR = CTX.log_dir

# Backward-compatible aliases used by older cells in this notebook.
DERIVED_OUTPUT_DIR = OUTPUT_DATA_DIR
FIGURE_DIR = OUTPUT_FIGURE_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output: {OUTPUT_DIR}")


Project root: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/falcon9_refactored_project
Notebook output: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/falcon9_refactored_project/outputs/04_reconcile_manual_event_catalogs


### Workflow contract

- Configuration is loaded from `config/project.yml`.
- This notebook writes only to `04_reconcile_manual_event_catalogs/` under the configured output root.
- Output filenames carry the `04_` prefix where they are declared explicitly.
- Upstream products are read through the product registry in the YAML file where practical.
- Existing outputs are protected from accidental overwrite by default.


## Objectives

1. Verify that millisecond-level timing is preserved.
2. Study same-channel and global inter-pick spacing.
3. Reconstruct candidate events using transparent clustering rules.
4. Assign accepted HD1–HD3 `N` picks directly to the 157 catalogue windows.
5. Compare the 153- and 157-event catalogues chronologically.
6. Identify merged, split, overlapping, weakly supported, and unmatched
   events.
7. Produce a concise review list for later waveform plotting.

## 1. Imports and paths

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

LEGACY_EXPORT_DIR = CONFIG.path("legacy_export_dir")

PICK_FILE = LEGACY_EXPORT_DIR / "manual_arrival_picks.csv"
CATALOG_153_FILE = (
    LEGACY_EXPORT_DIR
    / "legacy_infrasound_event_catalogue.csv"
)
CATALOG_157_FILE = (
    LEGACY_EXPORT_DIR
    / "legacy_catalog_157_events.csv"
)

OUTPUT_DIR = CTX.output_dir
FIGURE_DIR = CTX.figure_dir
DERIVED_DIR = CTX.data_dir

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Legacy export directory: {LEGACY_EXPORT_DIR}")

Project root: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/falcon9_refactored_project
Legacy export directory: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/falcon9_refactored_project/metadata/legacy_export


## 2. Load picks and both legacy catalogues

Epoch-second columns are preferred because they preserve the original
sub-second timing without depending on CSV datetime formatting.

In [3]:
for required_file in [
    PICK_FILE,
    CATALOG_153_FILE,
    CATALOG_157_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required export not found: {required_file}\n"
            "Rerun consolidate_spacex_matlab_legacy_data_epoch_complete.m"
        )

picks_all = pd.read_csv(PICK_FILE)
catalog_153 = pd.read_csv(CATALOG_153_FILE)
catalog_157 = pd.read_csv(CATALOG_157_FILE)

required_pick_columns = {
    "pickIndex",
    "arrivalTimeEpochS",
    "channel",
    "phase",
}
missing = required_pick_columns.difference(picks_all.columns)
if missing:
    raise KeyError(
        f"Pick table is missing required columns: {sorted(missing)}"
    )

required_153_columns = {
    "eventNumber",
    "firstArrivalEpochS",
    "lastArrivalEpochS",
}
missing = required_153_columns.difference(catalog_153.columns)
if missing:
    raise KeyError(
        "153-event catalogue is missing required columns: "
        f"{sorted(missing)}"
    )

required_157_columns = {
    "catalogEventNumber",
    "onTimeEpochS",
    "offTimeEpochS",
}
missing = required_157_columns.difference(catalog_157.columns)
if missing:
    raise KeyError(
        "157-event catalogue is missing required columns: "
        f"{sorted(missing)}"
    )

picks_all["arrival_time"] = pd.to_datetime(
    picks_all["arrivalTimeEpochS"],
    unit="s",
    utc=True,
)
catalog_153["first_arrival_time"] = pd.to_datetime(
    catalog_153["firstArrivalEpochS"],
    unit="s",
    utc=True,
)
catalog_153["last_arrival_time"] = pd.to_datetime(
    catalog_153["lastArrivalEpochS"],
    unit="s",
    utc=True,
)
catalog_157["on_time"] = pd.to_datetime(
    catalog_157["onTimeEpochS"],
    unit="s",
    utc=True,
)
catalog_157["off_time"] = pd.to_datetime(
    catalog_157["offTimeEpochS"],
    unit="s",
    utc=True,
)

picks_all["channel"] = (
    picks_all["channel"]
    .astype(str)
    .str.upper()
    .str.strip()
)
picks_all["phase_normalized"] = (
    picks_all["phase"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

if "isDeleted" in picks_all.columns:
    picks_all["is_deleted"] = (
        picks_all["isDeleted"]
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )
else:
    picks_all["is_deleted"] = (
        picks_all["phase_normalized"] == "del"
    )

print(f"Manual picks: {len(picks_all)}")
print(f"153-event catalogue: {len(catalog_153)}")
print(f"157-event catalogue: {len(catalog_157)}")

FileNotFoundError: Required export not found: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/falcon9_refactored_project/metadata/legacy_export/manual_arrival_picks.csv
Rerun consolidate_spacex_matlab_legacy_data_epoch_complete.m

## 3. Define the accepted infrasound-pick working set

In [ ]:
# Use the full temporal extent of the 157-event catalogue, with a small
# margin, rather than a hard-coded accident interval.
margin_s = 1.0
analysis_start_epoch_s = (
    catalog_157["onTimeEpochS"].min() - margin_s
)
analysis_end_epoch_s = (
    catalog_157["offTimeEpochS"].max() + margin_s
)

accepted_infrasound = picks_all.loc[
    picks_all["arrivalTimeEpochS"].between(
        analysis_start_epoch_s,
        analysis_end_epoch_s,
        inclusive="both",
    )
    & picks_all["channel"].isin(["HD1", "HD2", "HD3"])
    & picks_all["phase_normalized"].eq("n")
    & ~picks_all["is_deleted"]
].copy()

accepted_infrasound = accepted_infrasound.sort_values(
    ["arrivalTimeEpochS", "channel", "pickIndex"]
).reset_index(drop=True)

print(
    "Accepted HD1–HD3 N picks in the 157-catalogue interval:",
    len(accepted_infrasound),
)
display(
    accepted_infrasound[
        [
            "pickIndex",
            "arrival_time",
            "arrivalTimeEpochS",
            "channel",
            "phase",
        ]
    ].head(12)
)

## 4. Verify sub-second precision

This is a direct check that epoch-second values preserve the millisecond
timing seen in the MATLAB catalog display.

In [ ]:
precision_check = accepted_infrasound[
    [
        "pickIndex",
        "arrivalTimeEpochS",
        "arrival_time",
        "channel",
    ]
].head(20).copy()

precision_check["fractional_second"] = (
    precision_check["arrivalTimeEpochS"] % 1.0
)

display(precision_check)
print(
    "Unique fractional seconds among accepted picks:",
    accepted_infrasound["arrivalTimeEpochS"]
    .mod(1.0)
    .round(6)
    .nunique(),
)

## 5. Same-channel inter-pick spacing

In [ ]:
same_channel_parts = []

for channel, group in accepted_infrasound.groupby("channel"):
    group = group.sort_values("arrivalTimeEpochS").copy()
    group["same_channel_gap_s"] = (
        group["arrivalTimeEpochS"].diff()
    )
    same_channel_parts.append(group)

picks_with_same_channel_gaps = pd.concat(
    same_channel_parts,
    ignore_index=True,
).sort_values(["channel", "arrivalTimeEpochS"])

same_channel_gap_summary = (
    picks_with_same_channel_gaps
    .dropna(subset=["same_channel_gap_s"])
    .groupby("channel")["same_channel_gap_s"]
    .agg(
        count="count",
        minimum_s="min",
        p01_s=lambda x: x.quantile(0.01),
        p05_s=lambda x: x.quantile(0.05),
        p10_s=lambda x: x.quantile(0.10),
        median_s="median",
        p90_s=lambda x: x.quantile(0.90),
        p95_s=lambda x: x.quantile(0.95),
        maximum_s="max",
    )
    .reset_index()
)

display(same_channel_gap_summary)

same_channel_gap_summary.to_csv(
    DERIVED_DIR / "same_channel_gap_summary.csv",
    index=False,
)
picks_with_same_channel_gaps.to_csv(
    DERIVED_DIR / "accepted_picks_with_same_channel_gaps.csv",
    index=False,
)

In [ ]:
gap_values = (
    picks_with_same_channel_gaps["same_channel_gap_s"]
    .dropna()
)
gap_values = gap_values[gap_values > 0]

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.hist(
    gap_values,
    bins=np.logspace(
        np.log10(gap_values.min()),
        np.log10(gap_values.max()),
        80,
    ),
)
ax.set_xscale("log")
ax.set_xlabel("Time since previous pick on same channel (s)")
ax.set_ylabel("Number of picks")
ax.set_title("Same-channel inter-pick spacing")
ax.grid(True, which="both", alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("same_channel_gap_histogram.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Global inter-pick spacing

In [ ]:
accepted_sorted = accepted_infrasound.sort_values(
    ["arrivalTimeEpochS", "channel", "pickIndex"]
).reset_index(drop=True)

accepted_sorted["global_gap_s"] = (
    accepted_sorted["arrivalTimeEpochS"].diff()
)

display(
    accepted_sorted["global_gap_s"]
    .dropna()
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

accepted_sorted.to_csv(
    DERIVED_DIR / "accepted_infrasound_picks_with_global_gaps.csv",
    index=False,
)

In [ ]:
global_gaps = accepted_sorted["global_gap_s"].dropna()
global_gaps = global_gaps[global_gaps > 0]

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.hist(
    global_gaps,
    bins=np.logspace(
        np.log10(global_gaps.min()),
        np.log10(global_gaps.max()),
        100,
    ),
)
ax.set_xscale("log")
ax.axvline(0.1, linestyle="--", linewidth=1.2, label="0.1 s")
ax.set_xlabel("Time since previous accepted pick (s)")
ax.set_ylabel("Number of gaps")
ax.set_title("Global inter-pick spacing")
ax.grid(True, which="both", alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("global_pick_gap_histogram.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Transparent clustering-rule comparison

In [ ]:
def cluster_by_consecutive_gap(
    picks: pd.DataFrame,
    threshold_s: float,
) -> pd.DataFrame:
    work = picks.sort_values(
        ["arrivalTimeEpochS", "channel", "pickIndex"]
    ).copy()
    gaps = work["arrivalTimeEpochS"].diff()
    work["cluster_id"] = (
        gaps.isna() | gaps.gt(threshold_s)
    ).cumsum()
    return work


def cluster_by_first_pick_window(
    picks: pd.DataFrame,
    threshold_s: float,
) -> pd.DataFrame:
    work = picks.sort_values(
        ["arrivalTimeEpochS", "channel", "pickIndex"]
    ).copy()

    cluster_ids = np.empty(len(work), dtype=int)
    cluster_id = 0
    cluster_start = None

    for i, pick_epoch_s in enumerate(work["arrivalTimeEpochS"]):
        if cluster_start is None:
            cluster_id += 1
            cluster_start = pick_epoch_s
        elif pick_epoch_s - cluster_start > threshold_s:
            cluster_id += 1
            cluster_start = pick_epoch_s

        cluster_ids[i] = cluster_id

    work["cluster_id"] = cluster_ids
    return work


def summarize_clusters(
    clustered_picks: pd.DataFrame,
    minimum_distinct_channels: int = 2,
) -> pd.DataFrame:
    summary = (
        clustered_picks
        .groupby("cluster_id")
        .agg(
            first_pick_epoch_s=("arrivalTimeEpochS", "min"),
            last_pick_epoch_s=("arrivalTimeEpochS", "max"),
            pick_count=("pickIndex", "count"),
            distinct_channel_count=("channel", "nunique"),
            channels=(
                "channel",
                lambda x: ",".join(sorted(set(x))),
            ),
            pick_indices=(
                "pickIndex",
                lambda x: ",".join(map(str, x)),
            ),
        )
        .reset_index()
    )

    summary["duration_s"] = (
        summary["last_pick_epoch_s"]
        - summary["first_pick_epoch_s"]
    )
    summary["first_pick_time"] = pd.to_datetime(
        summary["first_pick_epoch_s"],
        unit="s",
        utc=True,
    )
    summary["last_pick_time"] = pd.to_datetime(
        summary["last_pick_epoch_s"],
        unit="s",
        utc=True,
    )
    summary["qualifies_as_event"] = (
        summary["distinct_channel_count"]
        >= minimum_distinct_channels
    )
    return summary

In [ ]:
thresholds_s = np.round(
    np.arange(0.02, 0.301, 0.002),
    3,
)

sweep_rows = []

algorithms = {
    "consecutive_gap": cluster_by_consecutive_gap,
    "first_pick_window": cluster_by_first_pick_window,
}

for algorithm_name, algorithm in algorithms.items():
    for threshold_s in thresholds_s:
        clustered = algorithm(
            accepted_infrasound,
            threshold_s,
        )
        summary = summarize_clusters(clustered)
        qualifying = summary.loc[
            summary["qualifies_as_event"]
        ]

        sweep_rows.append({
            "algorithm": algorithm_name,
            "threshold_s": threshold_s,
            "total_clusters": len(summary),
            "events_at_least_two_channels": len(qualifying),
            "events_all_three_channels": int(
                (summary["distinct_channel_count"] == 3).sum()
            ),
            "median_event_duration_s": (
                qualifying["duration_s"].median()
            ),
            "maximum_event_duration_s": (
                qualifying["duration_s"].max()
            ),
        })

threshold_sweep = pd.DataFrame(sweep_rows)

closest_to_legacy_counts = (
    threshold_sweep
    .assign(
        difference_from_153=lambda x: (
            x["events_at_least_two_channels"] - 153
        ).abs(),
        difference_from_157=lambda x: (
            x["events_at_least_two_channels"] - 157
        ).abs(),
    )
    .sort_values(
        [
            "difference_from_157",
            "difference_from_153",
            "algorithm",
            "threshold_s",
        ]
    )
    .head(30)
)

display(closest_to_legacy_counts)

threshold_sweep.to_csv(
    DERIVED_DIR / "clustering_threshold_sweep.csv",
    index=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.8))

for algorithm_name, group in threshold_sweep.groupby("algorithm"):
    ax.plot(
        group["threshold_s"],
        group["events_at_least_two_channels"],
        linewidth=1.2,
        label=algorithm_name.replace("_", " "),
    )

ax.axhline(153, linestyle="--", linewidth=1.0, label="Legacy 153")
ax.axhline(157, linestyle="-.", linewidth=1.0, label="Legacy 157")
ax.axvline(0.1, linestyle=":", linewidth=1.0, label="0.1 s")

ax.set_xlabel("Clustering threshold (s)")
ax.set_ylabel("Clusters with at least two channels")
ax.set_title("Candidate event count versus clustering threshold")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("event_count_vs_clustering_threshold.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. Assign accepted picks directly to the 157 catalogue windows

Picks are matched using epoch seconds and a very small boundary tolerance.
Overlapping windows are not silently resolved; they are flagged.

In [ ]:
CATALOG_BOUNDARY_TOLERANCE_S = 0.001

catalog_157_windows = catalog_157[
    [
        "catalogEventNumber",
        "onTimeEpochS",
        "offTimeEpochS",
        "on_time",
        "off_time",
        "durationS",
    ]
].copy()

pick_assignment_rows = []

for pick in accepted_infrasound.itertuples(index=False):
    matches = catalog_157_windows.loc[
        (
            pick.arrivalTimeEpochS
            >= catalog_157_windows["onTimeEpochS"]
            - CATALOG_BOUNDARY_TOLERANCE_S
        )
        & (
            pick.arrivalTimeEpochS
            <= catalog_157_windows["offTimeEpochS"]
            + CATALOG_BOUNDARY_TOLERANCE_S
        )
    ]

    if len(matches) == 0:
        status = "unmatched"
        matched_event_numbers = ""
    elif len(matches) == 1:
        status = "unique"
        matched_event_numbers = str(
            int(matches.iloc[0]["catalogEventNumber"])
        )
    else:
        status = "ambiguous_overlap"
        matched_event_numbers = ",".join(
            matches["catalogEventNumber"]
            .astype(int)
            .astype(str)
        )

    pick_assignment_rows.append({
        "pickIndex": pick.pickIndex,
        "arrivalTimeEpochS": pick.arrivalTimeEpochS,
        "arrival_time": pick.arrival_time,
        "channel": pick.channel,
        "catalog157_assignment_status": status,
        "catalog157_event_numbers": matched_event_numbers,
        "catalog157_match_count": len(matches),
    })

pick_assignments_157 = pd.DataFrame(pick_assignment_rows)

display(
    pick_assignments_157[
        "catalog157_assignment_status"
    ].value_counts()
)

pick_assignments_157.to_csv(
    DERIVED_DIR / "accepted_pick_assignments_to_catalog157.csv",
    index=False,
)

## 9. Summarize pick support for every 157-event window

In [ ]:
event_support_rows = []

for event in catalog_157_windows.itertuples(index=False):
    event_picks = accepted_infrasound.loc[
        (
            accepted_infrasound["arrivalTimeEpochS"]
            >= event.onTimeEpochS - CATALOG_BOUNDARY_TOLERANCE_S
        )
        & (
            accepted_infrasound["arrivalTimeEpochS"]
            <= event.offTimeEpochS + CATALOG_BOUNDARY_TOLERANCE_S
        )
    ].copy()

    event_support_rows.append({
        "catalog157_event_number": int(event.catalogEventNumber),
        "on_time_epoch_s": event.onTimeEpochS,
        "off_time_epoch_s": event.offTimeEpochS,
        "on_time": event.on_time,
        "off_time": event.off_time,
        "catalog_duration_s": event.durationS,
        "pick_count": len(event_picks),
        "distinct_channel_count": event_picks["channel"].nunique(),
        "channels": ",".join(
            sorted(event_picks["channel"].unique())
        ),
        "first_pick_epoch_s": (
            event_picks["arrivalTimeEpochS"].min()
            if len(event_picks)
            else np.nan
        ),
        "last_pick_epoch_s": (
            event_picks["arrivalTimeEpochS"].max()
            if len(event_picks)
            else np.nan
        ),
    })

catalog_157_support = pd.DataFrame(event_support_rows)

catalog_157_support["pick_span_s"] = (
    catalog_157_support["last_pick_epoch_s"]
    - catalog_157_support["first_pick_epoch_s"]
)
catalog_157_support["first_pick_minus_on_s"] = (
    catalog_157_support["first_pick_epoch_s"]
    - catalog_157_support["on_time_epoch_s"]
)
catalog_157_support["last_pick_minus_off_s"] = (
    catalog_157_support["last_pick_epoch_s"]
    - catalog_157_support["off_time_epoch_s"]
)
catalog_157_support["has_at_least_two_channels"] = (
    catalog_157_support["distinct_channel_count"] >= 2
)
catalog_157_support["has_all_three_channels"] = (
    catalog_157_support["distinct_channel_count"] == 3
)

display(
    catalog_157_support[
        [
            "pick_count",
            "distinct_channel_count",
            "has_at_least_two_channels",
            "has_all_three_channels",
        ]
    ].describe()
)

display(
    catalog_157_support.loc[
        ~catalog_157_support["has_at_least_two_channels"]
    ]
)

catalog_157_support.to_csv(
    DERIVED_DIR / "catalog157_event_pick_support.csv",
    index=False,
)

## 10. Detect overlapping 157-event windows

In [ ]:
catalog_157_sorted = catalog_157_windows.sort_values(
    "onTimeEpochS"
).reset_index(drop=True)

overlap_rows = []

for i in range(len(catalog_157_sorted) - 1):
    current = catalog_157_sorted.iloc[i]
    following = catalog_157_sorted.iloc[i + 1]

    overlap_s = (
        current["offTimeEpochS"]
        - following["onTimeEpochS"]
    )

    if overlap_s >= 0:
        overlap_rows.append({
            "event_a": int(current["catalogEventNumber"]),
            "event_b": int(following["catalogEventNumber"]),
            "event_a_on": current["on_time"],
            "event_a_off": current["off_time"],
            "event_b_on": following["on_time"],
            "event_b_off": following["off_time"],
            "overlap_s": overlap_s,
        })

catalog_157_overlaps = pd.DataFrame(overlap_rows)

print(
    "Number of overlapping/touching adjacent 157-event windows:",
    len(catalog_157_overlaps),
)
display(catalog_157_overlaps)

catalog_157_overlaps.to_csv(
    DERIVED_DIR / "catalog157_overlapping_windows.csv",
    index=False,
)

## 11. Compare the 153- and 157-event catalogues

Each 153-event interval is compared with all 157-event intervals. The
overlap duration and midpoint separation are used to identify likely
one-to-one matches, splits, and merges.

In [ ]:
catalog_153_compare = catalog_153[
    [
        "eventNumber",
        "firstArrivalEpochS",
        "lastArrivalEpochS",
        "first_arrival_time",
        "last_arrival_time",
    ]
].copy()

catalog_153_compare["midpoint_epoch_s"] = (
    catalog_153_compare["firstArrivalEpochS"]
    + catalog_153_compare["lastArrivalEpochS"]
) / 2.0

catalog_157_compare = catalog_157_windows.copy()
catalog_157_compare["midpoint_epoch_s"] = (
    catalog_157_compare["onTimeEpochS"]
    + catalog_157_compare["offTimeEpochS"]
) / 2.0

pair_rows = []

for event153 in catalog_153_compare.itertuples(index=False):
    for event157 in catalog_157_compare.itertuples(index=False):
        overlap_start = max(
            event153.firstArrivalEpochS,
            event157.onTimeEpochS,
        )
        overlap_end = min(
            event153.lastArrivalEpochS,
            event157.offTimeEpochS,
        )
        overlap_s = max(0.0, overlap_end - overlap_start)

        midpoint_difference_s = (
            event157.midpoint_epoch_s
            - event153.midpoint_epoch_s
        )

        if overlap_s > 0 or abs(midpoint_difference_s) <= 0.25:
            pair_rows.append({
                "event153": int(event153.eventNumber),
                "event157": int(event157.catalogEventNumber),
                "overlap_s": overlap_s,
                "midpoint_difference_s": midpoint_difference_s,
                "start_difference_s": (
                    event157.onTimeEpochS
                    - event153.firstArrivalEpochS
                ),
                "end_difference_s": (
                    event157.offTimeEpochS
                    - event153.lastArrivalEpochS
                ),
            })

catalogue_pairs = pd.DataFrame(pair_rows)

# Best 157 match for each 153 event:
best_157_for_153 = (
    catalogue_pairs
    .sort_values(
        [
            "event153",
            "overlap_s",
            "midpoint_difference_s",
        ],
        ascending=[True, False, True],
    )
    .groupby("event153", as_index=False)
    .first()
)

# Best 153 match for each 157 event:
best_153_for_157 = (
    catalogue_pairs
    .assign(
        absolute_midpoint_difference_s=lambda x: (
            x["midpoint_difference_s"].abs()
        )
    )
    .sort_values(
        [
            "event157",
            "overlap_s",
            "absolute_midpoint_difference_s",
        ],
        ascending=[True, False, True],
    )
    .groupby("event157", as_index=False)
    .first()
)

display(best_157_for_153.head(20))
display(best_153_for_157.head(20))

catalogue_pairs.to_csv(
    DERIVED_DIR / "catalog153_catalog157_candidate_pairs.csv",
    index=False,
)
best_157_for_153.to_csv(
    DERIVED_DIR / "catalog153_best_catalog157_match.csv",
    index=False,
)
best_153_for_157.to_csv(
    DERIVED_DIR / "catalog157_best_catalog153_match.csv",
    index=False,
)

## 12. Identify likely splits and merges

In [ ]:
meaningful_pairs = catalogue_pairs.loc[
    (catalogue_pairs["overlap_s"] > 0)
    | (catalogue_pairs["midpoint_difference_s"].abs() <= 0.05)
].copy()

split_counts = (
    meaningful_pairs
    .groupby("event153")["event157"]
    .nunique()
    .rename("n_catalog157_matches")
    .reset_index()
)
likely_splits = split_counts.loc[
    split_counts["n_catalog157_matches"] > 1
]

merge_counts = (
    meaningful_pairs
    .groupby("event157")["event153"]
    .nunique()
    .rename("n_catalog153_matches")
    .reset_index()
)
likely_merges = merge_counts.loc[
    merge_counts["n_catalog153_matches"] > 1
]

print("Possible 153 → multiple 157 splits:")
display(likely_splits)

print("Possible multiple 153 → one 157 merges:")
display(likely_merges)

likely_splits.to_csv(
    DERIVED_DIR / "possible_catalog153_splits.csv",
    index=False,
)
likely_merges.to_csv(
    DERIVED_DIR / "possible_catalog157_merges.csv",
    index=False,
)

## 13. Build the focused waveform-review list

An event is flagged for visual review when any of the following is true:

- fewer than two infrasound channels support a 157-event window;
- the 157 window overlaps or touches another window;
- the event participates in a possible split or merge;
- no close counterpart exists in the 153 catalogue;
- the 153 and 157 window boundaries differ substantially.

In [ ]:
review_reasons = {}

def add_reason(event_number: int, reason: str) -> None:
    review_reasons.setdefault(event_number, set()).add(reason)

for row in catalog_157_support.itertuples(index=False):
    event_number = int(row.catalog157_event_number)

    if row.distinct_channel_count < 2:
        add_reason(event_number, "fewer than two infrasound channels")

    if row.pick_count == 0:
        add_reason(event_number, "no accepted infrasound picks")

    if np.isfinite(row.first_pick_minus_on_s):
        if abs(row.first_pick_minus_on_s) > 0.005:
            add_reason(event_number, "first pick differs from ontime")

    if np.isfinite(row.last_pick_minus_off_s):
        if abs(row.last_pick_minus_off_s) > 0.005:
            add_reason(event_number, "last pick differs from offtime")

if not catalog_157_overlaps.empty:
    for row in catalog_157_overlaps.itertuples(index=False):
        add_reason(int(row.event_a), "overlapping/touching window")
        add_reason(int(row.event_b), "overlapping/touching window")

split_event_157s = set(
    meaningful_pairs.loc[
        meaningful_pairs["event153"].isin(
            likely_splits["event153"]
        ),
        "event157",
    ].astype(int)
)
for event_number in split_event_157s:
    add_reason(event_number, "possible split from 153 catalogue")

for event_number in likely_merges["event157"].astype(int):
    add_reason(event_number, "possible merge relative to 153 catalogue")

matched_157 = set(best_153_for_157["event157"].astype(int))
for event_number in catalog_157["catalogEventNumber"].astype(int):
    if event_number not in matched_157:
        add_reason(event_number, "no close 153-event counterpart")

for row in best_153_for_157.itertuples(index=False):
    if abs(row.start_difference_s) > 0.05:
        add_reason(int(row.event157), "start differs from 153 by >0.05 s")
    if abs(row.end_difference_s) > 0.05:
        add_reason(int(row.event157), "end differs from 153 by >0.05 s")

review_rows = [
    {
        "catalog157_event_number": event_number,
        "reasons": "; ".join(sorted(reasons)),
        "reason_count": len(reasons),
    }
    for event_number, reasons in sorted(review_reasons.items())
]

waveform_review_list = pd.DataFrame(review_rows)

waveform_review_list = waveform_review_list.merge(
    catalog_157_support,
    on="catalog157_event_number",
    how="left",
)

waveform_review_list = waveform_review_list.sort_values(
    [
        "reason_count",
        "catalog157_event_number",
    ],
    ascending=[False, True],
)

print(
    "157-event windows recommended for visual review:",
    len(waveform_review_list),
)
display(waveform_review_list)

waveform_review_list.to_csv(
    DERIVED_DIR / "waveform_review_list.csv",
    index=False,
)

## 14. Pick raster with 157-event windows

In [ ]:
channel_y = {"HD1": 3, "HD2": 2, "HD3": 1}

raster = accepted_infrasound.copy()
raster["channel_y"] = raster["channel"].map(channel_y)

fig, ax = plt.subplots(figsize=(12.0, 4.5))

ax.scatter(
    raster["arrival_time"],
    raster["channel_y"],
    s=14,
    label="Accepted picks",
)

for event in catalog_157_windows.itertuples(index=False):
    ax.axvspan(
        event.on_time,
        event.off_time,
        alpha=0.08,
    )

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["HD3", "HD2", "HD1"])
ax.set_ylim(0.5, 3.5)
ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Infrasound channel")
ax.set_title("Accepted manual picks and 157-event catalogue windows")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, axis="x", alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("accepted_pick_raster_with_catalog157.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 15. Conclusions to draw after execution

The most defensible final event catalogue will be the one for which:

- event windows are supported by at least two distinct infrasound channels;
- on/off times closely match the preserved manual picks;
- split and merge decisions are consistent with the actual waveforms;
- closely spaced events remain distinguishable on all three channels;
- the event-construction rule can be described transparently.

The `waveform_review_list.csv` output defines the subset requiring direct
inspection. There is no need to plot all 157 events unless the review list
remains unexpectedly large.

04_explore_manual_pick_clustering_updated.ipynb
Scientifically important, despite “explore” in the title.
This notebook documents:
* the 153- versus 157-event discrepancy;
* accepted manual picks;
* threshold sensitivity;
* catalogue matches;
* possible splits and merges;
* events requiring waveform review.
That is a critical provenance step. Rename it:
04_reconcile_manual_event_catalogues.ipynb
The notebook correctly treats the comparison as diagnostic and does not silently choose one catalogue. It should eventually finish with a small authoritative decision table:
event_catalogue_decisions.csv
with fields such as:
* legacy event IDs;
* final event ID;
* keep/merge/split/reject;
* rationale;
* reviewer note.
Verdict: keep and rename.